# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the FAIR² mass spectrometry protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed (uncomment if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
We'll load metadata and records from the Croissant dataset using `mlcroissant`. The dataset is defined at the following URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print(f"Version: {getattr(metadata, 'version', 'N/A')}; Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Let's review available record sets and fields. For this notebook, **all entities are referenced by their `@id`** for clarity and reproducibility. 

The list of record sets and fields (by `@id`) will be extracted and displayed.

In [ ]:
# List all record set @ids and their fields
def get_record_sets_and_fields(dataset):
    record_sets = dataset.metadata.record_sets
    record_sets_info = {}
    for rs in record_sets:
        fields = rs.fields
        field_ids = [field.id for field in fields]
        record_sets_info[rs.id] = field_ids
    return record_sets_info

record_sets_info = get_record_sets_and_fields(dataset)

print('Record Sets and Their Field @ids:')
for rs_id, field_ids in record_sets_info.items():
    print(f"RecordSet @id: {rs_id}")
    print(f"  Field @ids: {field_ids}\n")

## 3. Data Extraction
Now we will load data from **each record set** using their `@id` into Pandas DataFrames. 

You can use the record set or field `@id` values listed above for selective analyses.

In [ ]:
# Load all record sets into DataFrames keyed by their @id
all_dataframes = {}

for record_set_id in record_sets_info:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"  Loaded shape: {df.shape}")
        all_dataframes[record_set_id] = df
    except Exception as e:
        print(f"  Error loading records: {e}\n")

# Preview the first record set and its columns
if len(all_dataframes):
    first_rs_id = list(all_dataframes.keys())[0]
    print(f"\nColumns for RecordSet {first_rs_id}:\n", all_dataframes[first_rs_id].columns.tolist())
    all_dataframes[first_rs_id].head()
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Let's pick a numeric field from one record set (use its `@id`), filter (e.g., proteins with at least a minimum peptide count, or with MW over a threshold), normalize, and optionally group.

> Update the `numeric_field_id`, `record_set_id`, and `group_field_id` below using actual `@id`s from your data above.

In [ ]:
# Example EDA: Update the values below with appropriate @id's for your dataset
# For illustration, we use hypothetical IDs:
record_set_id = list(all_dataframes.keys())[0]  # or set to e.g. 'cr:MainRecordSet'
numeric_field_id = record_sets_info[record_set_id][1] if len(record_sets_info[record_set_id]) > 1 else record_sets_info[record_set_id][0]
group_field_id = record_sets_info[record_set_id][0]  # e.g., often a descriptive/categorical field

df = all_dataframes[record_set_id]

# Show available fields and first rows
print('Available fields:')
print(df.columns.tolist())
print(df.head())

# Try numeric analysis -- make sure field is numeric
if numeric_field_id in df.columns:
    try:
        df_numeric = df.copy()
        df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')
        threshold = df_numeric[numeric_field_id].quantile(0.75)  # Filter for top quartile
        filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print(f"Field {group_field_id} not in DataFrame columns for grouping.")
    except Exception as e:
        print(f"Could not process numeric EDA: {e}")
else:
    print(f"Field {numeric_field_id} not in DataFrame.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field, and the mean value by a group field (if applicable). We use seaborn/matplotlib for quick plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Group mean plot
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        group_means = group_means.sort_values(numeric_field_id, ascending=False)[:20]  # show top 20 if many
        plt.figure(figsize=(10, 5))
        sns.barplot(y=group_field_id, x=numeric_field_id, data=group_means)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print(f"Numeric field {numeric_field_id} could not be plotted.")

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant FAIR² mass spectrometry protein dataset metadata and records using only entity `@id` values,
- Inspect record sets and fields programmatically,
- Ingest each record set as a DataFrame, and
- Perform basic filtering, normalization, grouping, and visualization of quantitative mass spectrometry results.

Further downstream analyses (e.g. differential abundance, peptide/protein pattern discovery, or export for proteomics pipelines) can be implemented using the DataFrames and field `@id`s referenced above.